In [91]:
# ==============================================================
# CLASSIFICATION BINAIRE MULTICRITÈRE (PyTorch + M2 Pro GPU)
# ==============================================================
import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import re

from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
import joblib

In [92]:
# --------------------------------------------------------------
# 1️. CONFIGURATION GPU MPS
# --------------------------------------------------------------
print("="*70)
print("CONFIGURATION GPU APPLE M2 PRO")
print("="*70)
print(f"PyTorch version: {torch.__version__}")
print(f"MPS disponible: {torch.backends.mps.is_available()}")
print(f"MPS compilé: {torch.backends.mps.is_built()}")

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"✅ Utilisation du device : {device}")
print("="*70, "\n")

CONFIGURATION GPU APPLE M2 PRO
PyTorch version: 2.4.1
MPS disponible: True
MPS compilé: True
✅ Utilisation du device : mps



### Étape A — Ingestion + typage + nettoyage

#### Ingestion 

In [93]:
# 2. CHARGEMENT DU FICHIER ------------------------------------------------
df_fact_acorig = pd.read_csv("data/fisprod_acorig.csv", sep=',', header=0)
df_fact_valide= pd.read_csv("data/fisprod_valide.csv", sep=',', header=0)
df = pd.concat([df_fact_valide, df_fact_acorig], ignore_index=True, axis=0)
data=df.copy()
colonnes_all=df.columns
#df.head

In [94]:
df['observations_verification'].value_counts()

observations_verification
Quantité anormale d’acte Réduire la quantité à 1                                                                                                                                                          9495
RAS                                                                                                                                                                                                       7330
Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1                                                                                                         255
Bien et service non éligible pour la catégorie de patient                                                                                                                                                  170
L’acte ne correspond pas avec le sexe du patient                                                                                                  

In [95]:
# ---------------------------------------------------------------------
# Corriger le doublon exact dans observations_verification
# ---------------------------------------------------------------------

bad1 = "Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1"
bad2 = "Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1"
bad3 = "Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1"
good1 = "Quantité anormale d’acte Réduire la quantité à 1"

bad4 = "Bien et service non éligible pour la catégorie de patient; Quantité anormale d’acte Réduire la quantité à 1; Quantité anormale d’acte Réduire la quantité à 1"
bad7 = "Bien et service non éligible pour la catégorie de patient; Quantité anormale d’acte Réduire la quantité à 1"
good2 = "Bien et service non éligible pour la catégorie de patient"

bad5 = "L’acte ne correspond pas avec le sexe du patient; Bien et service non éligible pour la catégorie de patient"
bad6 = "L’acte ne correspond pas avec le sexe du patient; Quantité anormale d’acte Réduire la quantité à 1"
good3 = "L’acte ne correspond pas avec le sexe du patient"

bad8 = "Chevauchement détecté entre soin_intensif (2025-08-27 au 2025-08-28) et salle_standard (2025-08-26 au 2025-08-29)"
good4 = "Chevauchement de date d'hospitalisation détecté "

# Remplacement exact (pas de sous-chaîne, pas de regex partielle)
df['observations_verification'] = df['observations_verification'].replace(bad1, good1)
df['observations_verification'] = df['observations_verification'].replace(bad2, good1)
df['observations_verification'] = df['observations_verification'].replace(bad3, good1)

df['observations_verification'] = df['observations_verification'].replace(bad4, good2)
df['observations_verification'] = df['observations_verification'].replace(bad7, good2)

df['observations_verification'] = df['observations_verification'].replace(bad5, good3)
df['observations_verification'] = df['observations_verification'].replace(bad6, good3)

df['observations_verification'] = df['observations_verification'].replace(bad8, good4)


# Vérification du résultat
df['observations_verification'].value_counts()

observations_verification
Quantité anormale d’acte Réduire la quantité à 1                                                9767
RAS                                                                                             7330
Bien et service non éligible pour la catégorie de patient                                        175
L’acte ne correspond pas avec le sexe du patient                                                  44
Absence de: registre_number                                                                        6
Montant d'évacuation anormal                                                                       4
Prestation non éligible pour la Planification familiale avec type de prestation Sayana Press       1
Chevauchement de date d'hospitalisation détecté                                                    1
Facture saisie avant date d'entrée; Date de sortie est avant la date d’entrée                      1
Absence de: nom_patient                                          

In [96]:
# pour chaque valeur d'`observations_verification` ne garder que
# jusqu'à 600 factures dans un nouveau DataFrame
df_tronque = (
    df
    .groupby('observations_verification', group_keys=False)
    .apply(lambda g: g.head(600))
    .reset_index(drop=True)
)

# contrôle du résultat
print("taille du nouveau dataframe :", df_tronque.shape)
print(df_tronque['observations_verification'].value_counts())

taille du nouveau dataframe : (1433, 62)
observations_verification
Quantité anormale d’acte Réduire la quantité à 1                                                600
RAS                                                                                             600
Bien et service non éligible pour la catégorie de patient                                       175
L’acte ne correspond pas avec le sexe du patient                                                 44
Absence de: registre_number                                                                       6
Montant d'évacuation anormal                                                                      4
Absence de: nom_patient                                                                           1
Chevauchement de date d'hospitalisation détecté                                                   1
Facture saisie avant date d'entrée; Date de sortie est avant la date d’entrée                     1
Prestation non éligible pour la P

/var/folders/jy/94_gbknx2b300bycv73vb1jh0000gn/T/ipykernel_52553/1652999253.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.head(600))


In [97]:
df=df_tronque.copy()
df.shape

(1433, 62)

In [98]:
colonnes_all

Index(['id_structure', 'nom_patient', 'village', 'distance_village',
       'age_patient', 'sex', 'parent_name', 'tel', 'visit_date',
       'serie_number', 'registre_number', 'consultation_type',
       'type_prestation', 'num_ordonance', 'quantite_total_prod',
       'quantite_total_act', 'quantite_total_ex', 'cout_total_prod',
       'cout_total_act', 'cout_total_ex', 'type_observation', 'nbre_jours',
       'cout_mise_en_observation', 'cout_evacuation', 'nbre_kilomettre',
       'user_id', 'is_delete', 'id_user_created', 'id_user_updated',
       'id_user_deleted', 'deleted_at', 'created_at', 'updated_at',
       'assurance', 'taux_assurance', 'id', 'structure_assurance',
       'id_prescripteur', 'id_gerant', 'id_village', 'date_entree',
       'date_sortie', 'mode_sortie', 'data_source', 'id_api',
       'mobile_app_version', 'status_verification',
       'observations_verification', 'date_verification', 'facture_trouvee',
       'montant_contre_verification', 'facture_conforme',

In [99]:
df['observations_verification'].value_counts()

observations_verification
Quantité anormale d’acte Réduire la quantité à 1                                                600
RAS                                                                                             600
Bien et service non éligible pour la catégorie de patient                                       175
L’acte ne correspond pas avec le sexe du patient                                                 44
Absence de: registre_number                                                                       6
Montant d'évacuation anormal                                                                      4
Absence de: nom_patient                                                                           1
Chevauchement de date d'hospitalisation détecté                                                   1
Facture saisie avant date d'entrée; Date de sortie est avant la date d’entrée                     1
Prestation non éligible pour la Planification familiale avec type de prest

In [100]:
df['mode_sortie'].value_counts()

mode_sortie
59.0    1427
58.0       6
Name: count, dtype: int64

In [101]:
# Filter dfforml for records with specific libelle_type_prestation values
actes_cibles = ["Cryothérapie", "Césariennes","Laparotomie pour GEU","Résection à l’Anse Diathermique (RAD)"]

df_filtered = data[data['libelle_type_prestation'].isin(actes_cibles)].copy()

print(f"Nombre de lignes filtrées: {len(df_filtered)}")
print(f"\nActes trouvés:\n{df_filtered[['libelle_type_prestation', 'type_prestation']].value_counts()}")
print(f"\nShape du dataframe filtré: {df_filtered.shape}")

KeyError: 'libelle_type_prestation'

In [ ]:
print(df['libelle_type_structure'].unique())
print(f"\nNombre de valeurs uniques: {df['libelle_type_structure'].nunique()}")
print(f"\nComptage des valeurs:\n{df['libelle_type_structure'].value_counts()}")

['CSPS' 'CM' 'CMA' 'CHU' 'CHR']

Nombre de valeurs uniques: 5

Comptage des valeurs:
libelle_type_structure
CSPS    12174
CM       2030
CMA       458
CHR       177
CHU       166
Name: count, dtype: int64


In [ ]:
print(df['id_type_structure'].unique())
print(f"\nNombre de valeurs uniques: {df['id_type_structure'].nunique()}")
print(f"\nComptage des valeurs:\n{df['id_type_structure'].value_counts()}")

[43 42 46 44 45]

Nombre de valeurs uniques: 5

Comptage des valeurs:
id_type_structure
43    12174
42     2030
46      458
45      177
44      166
Name: count, dtype: int64


In [ ]:
print(df['libelle_type_prestation'].unique())
print(f"\nNombre de valeurs uniques: {df['libelle_type_prestation'].nunique()}")
print(f"\nComptage des valeurs:\n{df['libelle_type_prestation'].value_counts()}")

['Sayana Press' 'Soins préventifs' 'IVA/IVL'
 'Soins curatifs en ambulatoire grossesse'
 'Soins curatifs en ambulatoire enfant'
 'Soins curatifs en hospitalisation' 'Implant 5 ans insertion ou retrait'
 'Accouchements normaux' 'Pilule' 'Depo-Provera'
 'Résection à l’Anse Diathermique (RAD)' 'Soins curatifs du post partum'
 'DIU insertion ou retrait'
 "Accouchements assistés à l'aide d'instruments/produits"
 'Soins curatifs en interne (hospitalisation ou mise en observation)'
 'Implant 3 ans insertion ou retrait' 'Césariennes'
 'I.2.4.Actes (consultations)' 'Soins obstétricaux d’urgence'
 'Cryothérapie' 'Soins d’urgence aux nouveau-nés' 'Condom masculin'
 'Condom féminin' 'IV.2.4.Actes (consultations)' 'Laparotomie pour GEU']

Nombre de valeurs uniques: 25

Comptage des valeurs:
libelle_type_prestation
Soins curatifs en ambulatoire enfant                                  7127
Soins préventifs                                                      3041
Soins curatifs en ambulatoire grosses

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1433 entries, 0 to 1432
Data columns (total 62 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_structure                 1433 non-null   int64  
 1   nom_patient                  1432 non-null   object 
 2   village                      0 non-null      float64
 3   distance_village             1433 non-null   float64
 4   age_patient                  1433 non-null   object 
 5   sex                          1433 non-null   object 
 6   parent_name                  271 non-null    object 
 7   tel                          284 non-null    object 
 8   visit_date                   1433 non-null   object 
 9   serie_number                 377 non-null    object 
 10  registre_number              1427 non-null   object 
 11  consultation_type            1433 non-null   int64  
 12  type_prestation              1433 non-null   int64  
 13  num_ordonance     

In [103]:
colonnes_a_supprimer = [
       'id_structure',
       'parent_name', 'tel',
       'serie_number', 'num_ordonance', 'nbre_kilomettre',
       'user_id', 'is_delete', 'id_user_created', 'id_user_updated',
       'id_user_deleted', 'deleted_at', 'updated_at',
       'assurance', 'taux_assurance', 'id', 'structure_assurance',
       'id_village', 'data_source', 'id_api',
       'mobile_app_version', 'facture_trouvee',
       'montant_contre_verification', '_conforme',
       'is_contre_verification', 'id_contre_verification', 'observations',
       'numero_assurance', 'id_assurance', 'slug_assurance',
       'autre_type_prestation', 'id_motif', 'autre_motif',
       'qualite_accompagnant','facture_conforme',
       'libelle_consultation_type', 'user_name',
       'libelle_region', 'libelle_province', 'libelle_district',
       'libelle_commune','id_region', 'id_province',
       'id_district', 'id_commune', 'id_fs', 'libelle_mode_sortie',
       'id_niveau_structure',
       'is_public_structure', 'level_structure', 'date_debut', 'date_fin',
       'month_current', 'year_current', 'count_obs', 'count_ev'
]# Supprimer colonnes inutiles

set_all = set(colonnes_all)
set_sup = set(colonnes_a_supprimer)

communes = set_all & set_sup
seulement_all = set_all - set_sup
seulement_sup = set_sup - set_all

print("Colonnes communes :", communes)
print("Colonnes restantes :", seulement_all)
print("Colonnes inexistantes :", seulement_sup)

df.drop(columns=[c for c in colonnes_a_supprimer if c in df.columns], inplace=True)
print(" colonne de départ\n",data.columns)

Colonnes communes : {'deleted_at', 'numero_assurance', 'structure_assurance', 'serie_number', 'id_village', 'id', 'is_delete', 'id_user_deleted', 'tel', 'nbre_kilomettre', 'autre_motif', 'data_source', 'id_motif', 'updated_at', 'mobile_app_version', 'is_contre_verification', 'assurance', 'parent_name', 'facture_conforme', 'id_structure', 'qualite_accompagnant', 'id_api', 'autre_type_prestation', 'montant_contre_verification', 'user_id', 'facture_trouvee', 'id_user_updated', 'slug_assurance', 'taux_assurance', 'id_assurance', 'observations', 'num_ordonance', 'id_user_created', 'id_contre_verification'}
Colonnes restantes : {'date_entree', 'cout_total_prod', 'mode_sortie', 'cout_mise_en_observation', 'id_gerant', 'created_at', 'consultation_type', 'age_patient', 'quantite_total_act', 'sex', 'status_verification', 'quantite_total_prod', 'cout_evacuation', 'village', 'nbre_jours', 'observations_verification', 'date_verification', 'visit_date', 'registre_number', 'type_prestation', 'distanc

In [104]:
print("colonne retenue \n",df.columns)

colonne retenue 
 Index(['nom_patient', 'village', 'distance_village', 'age_patient', 'sex',
       'visit_date', 'registre_number', 'consultation_type', 'type_prestation',
       'quantite_total_prod', 'quantite_total_act', 'quantite_total_ex',
       'cout_total_prod', 'cout_total_act', 'cout_total_ex',
       'type_observation', 'nbre_jours', 'cout_mise_en_observation',
       'cout_evacuation', 'created_at', 'id_prescripteur', 'id_gerant',
       'date_entree', 'date_sortie', 'mode_sortie', 'status_verification',
       'observations_verification', 'date_verification'],
      dtype='object')


In [105]:
df.head()

,nom_patient,village,distance_village,age_patient,sex,visit_date,registre_number,consultation_type,type_prestation,quantite_total_prod,...,cout_evacuation,created_at,id_prescripteur,id_gerant,date_entree,date_sortie,mode_sortie,status_verification,observations_verification,date_verification
0,NaN,NaN,1.0,22-09-2006,female,2025-09-22,359,1,34,32.0,...,0.0,2025-09-23 12:06:01.000,27433,6071,NaN,NaN,59.0,a_corriger,Absence de: nom_patient,2025-10-11 13:14:03.000
1,KISWEFO RAFIATOU,NaN,1.0,15-09-2021,female,2025-09-15,NaN,4,20,25.0,...,0.0,2025-09-27 22:18:58.000,18582,4781,NaN,NaN,59.0,a_corriger,Absence de: registre_number,2025-10-11 13:14:15.000
2,OUALI TALAD,NaN,1.0,29-08-2024,female,2025-08-29,NaN,4,20,1.0,...,0.0,2025-09-01 06:53:05.000,16046,4159,NaN,NaN,59.0,a_corriger,Absence de: registre_number,2025-10-11 13:14:05.000
3,BAGAGNAN YASMINA,NaN,1.0,29-04-2024,female,2025-08-29,NaN,4,20,1.0,...,0.0,2025-09-13 07:40:57.000,14340,3697,NaN,NaN,59.0,a_corriger,Absence de: registre_number,2025-10-11 13:14:24.000
4,KISWEFO RAFIATOU,NaN,1.0,26-08-2021,female,2025-08-26,NaN,4,20,25.0,...,0.0,2025-09-24 03:43:15.000,18582,4781,NaN,NaN,59.0,a_corriger,Absence de: registre_number,2025-10-11 13:14:15.000


#### Nettoyage et typage

In [106]:
# ---------------------------
# 0) Paramètres
# provenance = village . obligatoire dans le formulaire csps pas pour CMU/CM/CHU/CHR
# ---------------------------
#  observations_verification : contie
# nt du texte utile
target_col = ["status_verification","observations_verification"]

#filled_cols pertinent pour déterminer la complétude des données ' tenir compte de "village",

filled_cols = [
    "nom_patient", "distance_village", "age_patient",
    "registre_number", "id_prescripteur", "id_gerant", "sex", "consultation_type","type_prestation",'id_type_structure', 'visit_date'
]

date_cols = ["date_entree", "date_sortie", "visit_date","created_at"]

num_cols = [
    "distance_village", "age_patient","nbre_jours",
    "quantite_total_prod", "quantite_total_act", "quantite_total_ex",
    "cout_total_prod", "cout_total_act", "cout_total_ex",
    "cout_mise_en_observation", "cout_evacuation","consultation_type", "type_prestation", "type_observation", "mode_sortie","id_type_structure"
]

cat_cols = [
     'libelle_fs', 'libelle_type_structure', 'libelle_type_prestation'
]

A) Règles FORMELLES (qualité de saisie)

In [107]:
# ==============================================================
# conversion des colonne à chiffre 
# ==============================================================
def to_float(x):
    """Convertit en float (gère '1 234,56', '1,234.56', etc.)."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)

    s = str(x).strip().lower()
    if s in {"", "none", "nan", "null"}:
        return np.nan

    s = re.sub(r"[^0-9\-,.]", "", s)

    # 1234,56 -> 1234.56
    if s.count(",") == 1 and s.count(".") == 0:
        s = s.replace(",", ".")

    # 1,234.56 -> 1234.56
    if s.count(",") > 0 and s.count(".") == 1:
        s = s.replace(",", "")

    try:
        return float(s)
    except:
        return np.nan
    

# Convert all numeric columns that might have mixed formats
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].apply(to_float)
        print(f"✓ {col} converted")

✓ distance_village converted
✓ age_patient converted
✓ nbre_jours converted
✓ quantite_total_prod converted
✓ quantite_total_act converted
✓ quantite_total_ex converted
✓ cout_total_prod converted
✓ cout_total_act converted
✓ cout_total_ex converted
✓ cout_mise_en_observation converted
✓ cout_evacuation converted
✓ consultation_type converted
✓ type_prestation converted
✓ type_observation converted
✓ mode_sortie converted


In [108]:
# ==============================================================
# TEST DE COMPLÉTUDE DES DONNÉES - filled_cols # champs obligatoires manquants
# ==============================================================

print("="*70)
print("ANALYSE DE COMPLÉTUDE DES DONNÉES")
print("="*70)

# Afficher les statistiques de complétude
completude_stats = []
for col in filled_cols:
    if col in df.columns:
        non_null_count = df[col].notna().sum()
        null_count = df[col].isna().sum()
        completude_pct = (non_null_count / len(df)) * 100
        completude_stats.append({
            'Colonne': col,
            'Non-vides': non_null_count,
            'Vides': null_count,
            'Complétude %': f"{completude_pct:.2f}%"
        })
    else:
        print(f"⚠️  Colonne '{col}' introuvable dans le dataframe")

completude_df = pd.DataFrame(completude_stats)
print("\n STATISTIQUES DE COMPLÉTUDE:\n")
print(completude_df.to_string(index=False))

# Créer les colonnes de flags (0 ou 1)
print("\n" + "="*70)
print("CRÉATION DES FLAGS DE COMPLÉTUDE")
print("="*70)

for col in filled_cols:
    if col in df.columns:
        flag_col = f"{col}_is_filled"
        df[flag_col] = df[col].notna().astype(int)
        filled_count = df[flag_col].sum()
        print(f"✓ {flag_col}: {filled_count} remplis, {len(df) - filled_count} vides")

print("\n Colonnes créées:")
print([c for c in df.columns if "_is_filled" in c])


ANALYSE DE COMPLÉTUDE DES DONNÉES
⚠️  Colonne 'id_type_structure' introuvable dans le dataframe

 STATISTIQUES DE COMPLÉTUDE:

          Colonne  Non-vides  Vides Complétude %
      nom_patient       1432      1       99.93%
 distance_village       1433      0      100.00%
      age_patient          0   1433        0.00%
  registre_number       1427      6       99.58%
  id_prescripteur       1433      0      100.00%
        id_gerant       1433      0      100.00%
              sex       1433      0      100.00%
consultation_type       1433      0      100.00%
  type_prestation       1433      0      100.00%
       visit_date       1433      0      100.00%

CRÉATION DES FLAGS DE COMPLÉTUDE
✓ nom_patient_is_filled: 1432 remplis, 1 vides
✓ distance_village_is_filled: 1433 remplis, 0 vides
✓ age_patient_is_filled: 0 remplis, 1433 vides
✓ registre_number_is_filled: 1427 remplis, 6 vides
✓ id_prescripteur_is_filled: 1433 remplis, 0 vides
✓ id_gerant_is_filled: 1433 remplis, 0 vides
✓ sex_i

In [109]:
# ==============================================================
# CONVERSION DE age_patient EN DATE DE NAISSANCE
# CALCUL DE L'ÂGE EN FONCTION DE L'ANNÉE EN COURS
# ==============================================================

print("="*70)
print("CONVERSION age_patient EN DATE + CALCUL ÂGE")
print("="*70)

# Référence de date
today = pd.Timestamp.today()
current_year = today.year

# CONVERTIR age_patient EN DATE (date de naissance)
# Convertir en datetime (format: jour/mois/année )
df['age_patient_date_naissance'] = pd.to_datetime(
    df['age_patient'], 
    errors='coerce', 
    dayfirst=True  
)

# CALCULER L'ÂGE EN FONCTION DE L'ANNÉE EN COURS
# Formule: Âge = Année actuelle - Année naissance
# (Approximation simple, sans tenir compte du jour/mois)
df['age_patient_calculated'] = df['age_patient_date_naissance'].dt.year.apply(
    lambda year: current_year - year if pd.notna(year) else np.nan
)

print(f"\n  Exemples de calculs d'âge:")
examples = df[df['age_patient_date_naissance'].notna()][
    ['age_patient', 'age_patient_date_naissance', 'age_patient_calculated']
].drop_duplicates().head(15)
print(examples.to_string())

CONVERSION age_patient EN DATE + CALCUL ÂGE

  Exemples de calculs d'âge:
Empty DataFrame
Columns: [age_patient, age_patient_date_naissance, age_patient_calculated]
Index: []


In [110]:
# ==============================================================
# VALIDATION DE L'ÂGE (calcul de l'age) ET DU SEXE
# ==============================================================

print("="*70)
print("VALIDATION DE L'ÂGE ET DU SEXE")
print("="*70)

# age_patient_is_valid = 1 si valide (entre 0 et 120), 0 sinon
df['age_patient_is_valid'] = (
    (df['age_patient_calculated'].notna()) & 
    (df['age_patient_calculated'] > 0) & 
    (df['age_patient_calculated'] <= 120)
).astype(int)

# Statistiques
age_valid_count = df['age_patient_is_valid'].sum()
age_invalid_count = len(df) - age_valid_count

print(f"\n ÂGE:")
print(f"  Valides (0 < age <= 120): {age_valid_count}")
print(f"  Invalides (null, <=0, >120): {age_invalid_count}")
#print(f"  Suspect (age > 100 ans): {age_suspect_count}")

# Afficher quelques exemples d'âges invalides
""" if age_invalid_count > 0:
    print(f"\n  Exemples d'âges invalides:")
    invalid_ages = df[df['age_patient_is_valid'] == 0][['age_patient', 'age_patient_calculated', 'visit_date', 'days_since_visit']].drop_duplicates()
    print(invalid_ages.head(10).to_string()) """

# Afficher les âges suspects
""" if age_suspect_count > 0:
    print(f"\n  Exemples d'âges suspects (> 100):")
    suspect_ages = df[df['age_patient_is_suspect'] == 1][['age_patient', 'age_patient_calculated', 'visit_date']].drop_duplicates()
    print(suspect_ages.head(10).to_string()) """

# 2) Validation du sexe
# sex_is_valid = 1 si valide (m, f, ou équivalent), 0 sinon
valid_sex_values = {'m', 'f', 'male', 'female', 'masculin', 'féminin', 'femme', 'homme', '1', '2'}
df['sex_is_valid'] = df['sex'].fillna('').str.strip().str.lower().isin(valid_sex_values).astype(int)

sex_valid_count = df['sex_is_valid'].sum()
sex_invalid_count = len(df) - sex_valid_count
print(f"\n SEXE:")
print(f"  Valides (m/f ou équivalent): {sex_valid_count}")
print(f"  Invalides (null, autre): {sex_invalid_count}")

# if sex_invalid_count > 0:
#     print(f"\n  Exemples de sexes invalides:")
#     invalid_sex = df[df['sex_is_valid'] == 0][['sex', 'sex_is_valid']].drop_duplicates()
#     print(invalid_sex.head(10).to_string())



VALIDATION DE L'ÂGE ET DU SEXE

 ÂGE:
  Valides (0 < age <= 120): 0
  Invalides (null, <=0, >120): 1433

 SEXE:
  Valides (m/f ou équivalent): 1433
  Invalides (null, autre): 0


B) Règles TEMPORELLES (très proches de ton PHP)


In [111]:
# ==============================================================
# CONVERSION DES DATES ET CRÉATION DES FLAGS DE VALIDATION
# ==============================================================

print("="*70)
print("CONVERSION DES DATES + FLAGS DE VALIDATION")
print("="*70)

# Convertir les dates et créer les flags
for col in date_cols:
    if col in df.columns:
        # Avant conversion
        print(f"\n{'─'*70}")
        print(f"Colonne: {col}")
        print(f"{'─'*70}")
        print(f"Avant: Type={df[col].dtype}, Non-vides={df[col].notna().sum()}, Vides={df[col].isna().sum()}")
        
        # Convertir en datetime (JJ/MM/AAAA)
        df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
        
        # Créer flag: 1 si valide, 0 si invalide/manquant
        flag_col = f"{col}_is_valid"
        df[flag_col] = df[col].notna().astype(int)
        
        # Stats
        valid_count = df[flag_col].sum()
        invalid_count = len(df) - valid_count
        
        print(f"Après:  Valides={valid_count}, Invalides={invalid_count}")
        print(f"Flag créé: {flag_col} (1=valide, 0=invalide)")
        
        # Exemples
        if valid_count > 0:
            print(f"  Exemples (valides): {df[df[col].notna()][col].head(3).dt.strftime('%d/%m/%Y').tolist()}")

print(f"\n{'='*70}")
print("RÉSUMÉ DES FLAGS CRÉÉS")
print(f"{'='*70}")
for col in date_cols:
    if col in df.columns:
        flag_col = f"{col}_is_valid"
        valid = df[flag_col].sum()
        invalid = len(df) - valid
        print(f"✓ {flag_col:25} | Valides: {valid:5} | Invalides: {invalid:5}")

print(f"\n✅ Dates converties et validées")


CONVERSION DES DATES + FLAGS DE VALIDATION

──────────────────────────────────────────────────────────────────────
Colonne: date_entree
──────────────────────────────────────────────────────────────────────
Avant: Type=object, Non-vides=147, Vides=1286
Après:  Valides=147, Invalides=1286
Flag créé: date_entree_is_valid (1=valide, 0=invalide)
  Exemples (valides): ['21/09/2025', '18/10/2025', '24/09/2025']

──────────────────────────────────────────────────────────────────────
Colonne: date_sortie
──────────────────────────────────────────────────────────────────────
Avant: Type=object, Non-vides=148, Vides=1285
Après:  Valides=148, Invalides=1285
Flag créé: date_sortie_is_valid (1=valide, 0=invalide)
  Exemples (valides): ['22/09/2025', '22/10/2025', '25/09/2025']

──────────────────────────────────────────────────────────────────────
Colonne: visit_date
──────────────────────────────────────────────────────────────────────
Avant: Type=object, Non-vides=1433, Vides=0
Après:  Valides=14

/var/folders/jy/94_gbknx2b300bycv73vb1jh0000gn/T/ipykernel_52553/714308089.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
/var/folders/jy/94_gbknx2b300bycv73vb1jh0000gn/T/ipykernel_52553/714308089.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
/var/folders/jy/94_gbknx2b300bycv73vb1jh0000gn/T/ipykernel_52553/714308089.py:19: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
/var/folders/jy/94_gbknx2b300bycv73vb1jh0000gn/T/ipykernel_52553/714308089.py:19: Use

In [ ]:
# # ==============================================================
# # AFFICHER LES DATES VIDES OU INVALIDES


# print("="*70)
# print("ANALYSE DES DATES VIDES OU INVALIDES")
# print("="*70)

# for col in date_cols:
#     if col in df.columns:
#         print(f"\n{'─'*70}")
#         print(f"Colonne: {col}")
#         print(f"{'─'*70}")
        
#         # Comptage
#         valid_count = df[col].notna().sum()
#         invalid_count = df[col].isna().sum()
#         total = len(df)
        
#         print(f"  Total: {total}")
#         print(f"  Valides (non-null): {valid_count} ({valid_count/total*100:.2f}%)")
#         print(f"  Invalides (null): {invalid_count} ({invalid_count/total*100:.2f}%)")
        
#         # Afficher quelques exemples de dates invalides
#         if invalid_count > 0:
#             print(f"\n  Exemples de lignes avec dates manquantes:")
#             invalid_examples = df[df[col].isna()][
#                 ['nom_patient', col, 'visit_date', 'status_verification']
#             ].head(10)
#             print(invalid_examples.to_string(index=True))

# print(f"\n{'='*70}")
# print("RÉSUMÉ DES DATES INVALIDES")
# print(f"{'='*70}")
# for col in date_cols:
#     if col in df.columns:
#         invalid = df[col].isna().sum()
#         pct = (invalid / len(df)) * 100
#         print(f"✗ {col:20} | Invalides: {invalid:5} ({pct:6.2f}%)")
        
# # ==============================================================

In [112]:
# ==============================================================
#  VÉRIFICATION: DATE D'ENTRÉE POSTÉRIEURE À LA DATE DE SAISIE (verifierIncoherenceDateEntree )
# ==============================================================

print("="*70)
print("VÉRIFICATION DATE ENTRÉE > DATE SAISIE (createdAt)")
print("="*70)

# Convertir les deux colonnes en datetime
df['date_entree'] = pd.to_datetime(df['date_entree'], errors='coerce', dayfirst=True)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', dayfirst=True)
    
# Logique:
# Si date_entree > created_at (entrée APRÈS la saisie du formulaire) → 1 (SUSPECT)
# Sinon → 0 (NORMAL)
    
df['verifierIncoherenceDateEntree'] = (
    (df['date_entree'].notna()) &  # date_entree existe
    (df['created_at'].notna()) &   # created_at existe
    (df['date_entree'] > df['created_at'])  # entrée APRÈS saisie = SUSPECT
).astype(int)
    
# Statistiques
suspect_count = df['verifierIncoherenceDateEntree'].sum()
normal_count = len(df) - suspect_count
    
print(f"\n RÉSULTAT:")
print(f"  ✓ Dates normales (entrée ≤ saisie): {normal_count}")
print(f"  ⚠️  Dates suspectes (entrée > saisie): {suspect_count}")
    
# Afficher les exemples suspects
if suspect_count > 0:
    print(f"\n  Exemples suspectes (dateEntree > createdAt):")
    suspect_examples = df[df['verifierIncoherenceDateEntree'] == 1][
        ['date_entree', 'created_at', 'verifierIncoherenceDateEntree']
    ].drop_duplicates().head(10)
    print(suspect_examples.to_string())
    
print(f"\n✅ Colonne créée: verifierIncoherenceDateEntree (0=normal, 1=suspect)")



VÉRIFICATION DATE ENTRÉE > DATE SAISIE (createdAt)

 RÉSULTAT:
  ✓ Dates normales (entrée ≤ saisie): 1432
  ⚠️  Dates suspectes (entrée > saisie): 1

  Exemples suspectes (dateEntree > createdAt):
    date_entree          created_at  verifierIncoherenceDateEntree
183  2025-10-18 2025-10-03 09:52:13                              1

✅ Colonne créée: verifierIncoherenceDateEntree (0=normal, 1=suspect)


In [113]:
# ==============================================================
# VÉRIFICATION: DATE DE SORTIE POSTÉRIEURE À LA DATE DE SAISIE (verifierIncoherenceDateSortie)
# ==============================================================

print("="*70)
print("VÉRIFICATION DATE SORTIE > DATE SAISIE (createdAt)")
print("="*70)

# Convertir les deux colonnes en datetime
df['date_sortie'] = pd.to_datetime(df['date_sortie'], errors='coerce', dayfirst=True)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', dayfirst=True)
    
# Logique:
# Si date_sortie > created_at (entrée APRÈS la saisie du formulaire) → 1 (SUSPECT)
# Sinon → 0 (NORMAL)
    
df['verifierIncoherenceDateSortie'] = (
    (df['date_sortie'].notna()) &  # date_sortie existe
    (df['created_at'].notna()) &   # created_at existe
    (df['date_sortie'] > df['created_at'])  # entrée APRÈS saisie = SUSPECT
).astype(int)
    
# Statistiques
suspect_count = df['verifierIncoherenceDateSortie'].sum()
normal_count = len(df) - suspect_count
    
print(f"\n RÉSULTAT:")
print(f"  ✓ Dates normales (sortie ≤ saisie): {normal_count}")
print(f"  ⚠️  Dates suspectes (sortie > saisie): {suspect_count}")
    
# Afficher les exemples suspects
if suspect_count > 0:
    print(f"\n  Exemples suspectes (datesortie > createdAt):")
    suspect_examples = df[df['verifierIncoherenceDateSortie'] == 1][
        ['date_sortie', 'created_at', 'verifierIncoherenceDateSortie']
    ].drop_duplicates().head(10)
    print(suspect_examples.to_string())
    
print(f"\n✅ Colonne créée: verifierIncoherenceDateSortie (0=normal, 1=suspect)")



VÉRIFICATION DATE SORTIE > DATE SAISIE (createdAt)

 RÉSULTAT:
  ✓ Dates normales (sortie ≤ saisie): 1432
  ⚠️  Dates suspectes (sortie > saisie): 1

  Exemples suspectes (datesortie > createdAt):
    date_sortie          created_at  verifierIncoherenceDateSortie
183  2025-10-22 2025-10-03 09:52:13                              1

✅ Colonne créée: verifierIncoherenceDateSortie (0=normal, 1=suspect)


In [114]:
# ==============================================================
# VÉRIFICATION DE L'INCOHÉRENCE DES DATES ENTRÉE/SORTIE (verifierChevauchementHospitalisation)
# ==============================================================

print("="*70)
print("VÉRIFICATION INCOHÉRENCE DATES ENTRÉE/SORTIE")
print("="*70)

# Logique:
# Si date_entree ET date_sortie sont non-null ET date_sortie < date_entree → 1 (INCOHÉRENT)
# Sinon → 0 (COHÉRENT ou données manquantes)

df['verifierChevauchementHospitalisation'] = (
    (df['date_entree'].notna()) &  # date_entree existe
    (df['date_sortie'].notna()) &  # date_sortie existe
    (df['date_sortie'] < df['date_entree'])  # sortie AVANT entrée = INCOHÉRENT
).astype(int)

# Statistiques
incoherent_count = df['verifierChevauchementHospitalisation'].sum()
coherent_count = len(df) - incoherent_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Dates cohérentes (ou manquantes): {coherent_count}")
print(f"  ✗ Dates incohérentes (sortie < entrée): {incoherent_count}")

# Afficher les exemples d'incohérence
if incoherent_count > 0:
    print(f"\n  Exemples d'incohérences (date_sortie < date_entree):")
    incoherent_examples = df[df['verifierChevauchementHospitalisation'] == 1][
        ['date_entree', 'date_sortie', 'verifierChevauchementHospitalisation']
    ].drop_duplicates().head(10)
    print(incoherent_examples.to_string())

print(f"\n Colonne créée: verifierChevauchementHospitalisation (0=cohérent, 1=incohérent)")


VÉRIFICATION INCOHÉRENCE DATES ENTRÉE/SORTIE

 RÉSULTAT:
  ✓ Dates cohérentes (ou manquantes): 1433
  ✗ Dates incohérentes (sortie < entrée): 0

 Colonne créée: verifierChevauchementHospitalisation (0=cohérent, 1=incohérent)


In [115]:
# ==============================================================
# DÉTECTION DES CHEVAUCHEMENTS DE DATES HOSPITALISATION (verifierChevauchementHospitalisation)
# Vérifier si un patient a des périodes
# d'hospitalisation qui se chevauchent (excluant les jours consécutifs)
# ==============================================================

print("="*70)
print("DÉTECTION DES CHEVAUCHEMENTS D'HOSPITALISATION")
print("="*70)

def detect_overlapping_hospitalizations(df, patient_id_col='nom_patient'):
    """
    Détecte les chevauchements de dates d'hospitalisation pour chaque patient.
    
    Logic:
    - Grouper par patient (nom_patient)
    - Pour chaque paire de périodes, vérifier le chevauchement
    - Exclure les cas où les périodes sont consécutives (sortie = entrée suivante)
    - Créer un flag: 1 si chevauchement détecté, 0 sinon
    """
    
    overlaps = []
    
    # Grouper par patient
    for patient, group in df.groupby(patient_id_col):
        # Si le patient n'a qu'une seule période, pas de chevauchement possible
        if len(group) <= 1:
            continue
        
        # Réinitialiser l'index pour accéder facilement aux lignes
        group = group.reset_index(drop=True)
        
        # Vérifier toutes les paires de périodes
        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                row1 = group.iloc[i]
                row2 = group.iloc[j]
                
                # Vérifier si les dates existent
                if pd.isna(row1['date_entree']) or pd.isna(row1['date_sortie']) or \
                   pd.isna(row2['date_entree']) or pd.isna(row2['date_sortie']):
                    continue
                
                # Récupérer les dates (au niveau du jour)
                entree1 = pd.Timestamp(row1['date_entree']).normalize()
                sortie1 = pd.Timestamp(row1['date_sortie']).normalize()
                entree2 = pd.Timestamp(row2['date_entree']).normalize()
                sortie2 = pd.Timestamp(row2['date_sortie']).normalize()
                
                # Vérifier chevauchement potentiel:
                # Si sortie1 >= entree2 ET entree1 <= sortie2
                if sortie1 >= entree2 and entree1 <= sortie2:
                    # Vérifier si ce sont juste des jours consécutifs
                    sont_consecutives = (sortie1 == entree2) or (sortie2 == entree1)
                    
                    # Vérifier si sortie est le jour suivant de l'entrée
                    jour_consecutif = (sortie1 + pd.Timedelta(days=1) == entree2) or \
                                     (sortie2 + pd.Timedelta(days=1) == entree1)
                    
                    # Si ce n'est pas juste consécutif → chevauchement!
                    if not sont_consecutives and not jour_consecutif:
                        overlaps.append({
                            'patient': patient,
                            'index1': group.index[i],
                            'index2': group.index[j],
                            'entree1': entree1,
                            'sortie1': sortie1,
                            'entree2': entree2,
                            'sortie2': sortie2,
                            'details': f"Chevauchement détecté: {entree1.strftime('%d/%m/%Y')} au {sortie1.strftime('%d/%m/%Y')} vs {entree2.strftime('%d/%m/%Y')} au {sortie2.strftime('%d/%m/%Y')}"
                        })
    
    return overlaps

# Détection des chevauchements
overlaps = detect_overlapping_hospitalizations(df, patient_id_col='nom_patient')

# Créer un flag: 1 si le patient est impliqué dans un chevauchement
df['verifierChevauchementHospitalisation'] = 0

for overlap in overlaps:
    idx1 = overlap['index1']
    idx2 = overlap['index2']
    df.loc[idx1, 'verifierChevauchementHospitalisation'] = 1
    df.loc[idx2, 'verifierChevauchementHospitalisation'] = 1

# Statistiques
overlap_count = df['verifierChevauchementHospitalisation'].sum()
no_overlap_count = len(df) - overlap_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Sans chevauchement: {no_overlap_count}")
print(f"  ⚠️  Avec chevauchement: {overlap_count}")

# Afficher les détails des chevauchements détectés
if overlaps:
    print(f"\n  Détails des {len(overlaps)} chevauchements détectés:")
    for idx, overlap in enumerate(overlaps[:10], 1):  # Afficher max 10
        print(f"\n  #{idx} - Patient: {overlap['patient']}")
        print(f"      {overlap['details']}")
else:
    print(f"\n  ✅ Aucun chevauchement d'hospitalisation détecté!")

print(f"\n✅ Colonne créée: verifierChevauchementHospitalisation (0=pas de chevauchement, 1=chevauchement)")

DÉTECTION DES CHEVAUCHEMENTS D'HOSPITALISATION

 RÉSULTAT:
  ✓ Sans chevauchement: 1428
  ⚠️  Avec chevauchement: 5

  Détails des 4 chevauchements détectés:

  #1 - Patient: DIAKITE ASSETA
      Chevauchement détecté: 23/09/2025 au 26/09/2025 vs 23/09/2025 au 26/09/2025

  #2 - Patient: OUEDRAOGO MARIAM
      Chevauchement détecté: 12/09/2025 au 15/09/2025 vs 14/09/2025 au 16/09/2025

  #3 - Patient: OUEDRAOGO MARIAM
      Chevauchement détecté: 14/09/2025 au 16/09/2025 vs 15/09/2025 au 16/09/2025

  #4 - Patient: SALGO MARIAM
      Chevauchement détecté: 05/09/2025 au 08/09/2025 vs 05/09/2025 au 08/09/2025

✅ Colonne créée: verifierChevauchementHospitalisation (0=pas de chevauchement, 1=chevauchement)


B) Règles CATEGORIELLE (qualité de saisie)

In [116]:
#actes_specialises = [
#    "Laparotomie pour GEU",14
#    "Césariennes", 12
#    "Cryothérapie", 17
#    "Résection à l'Anse Diathermique (RAD)" 18
#]
# ==============================================================
# VÉRIFICATION (verifierIncoherenceFormationSanitaire): 
# CSPS (id_type_structure = 43) bénéficiant d'actes spécialisés
# ==============================================================

print("="*70)
print("VÉRIFICATION CSPS BÉNÉFICIANT D'ACTES SPÉCIALISÉS")
print("="*70)

# Liste des actes spécialisés
actes_specialises = [
    "Laparotomie pour GEU",
    "Césariennes", 
    "Cryothérapie",
    "Résection à l'Anse Diathermique (RAD)" 
]

# Créer le flag basé sur les conditions:
# Si id_type_structure = 43 ET libelle_type_prestation IN (actes_specialises)
# → 1 (incohérence détectée), 0 sinon

df['verifierIncoherenceFormationSanitaire'] = (
    (df['id_type_structure'] == 43) &
    (df['libelle_type_prestation'].isin(actes_specialises))
).astype(int)

# Statistiques
anomaly_count = df['verifierIncoherenceFormationSanitaire'].sum()
normal_count = len(df) - anomaly_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Cas normaux (pas d'incohérence): {normal_count}")
print(f"  ⚠️  Incohérences détectées: {anomaly_count}")

# Afficher les exemples d'incohérences
if anomaly_count > 0:
    print(f"\n  Exemples (CSPS avec actes spécialisés):")
    anomaly_examples = df[df['verifierIncoherenceFormationSanitaire'] == 1][
        ['libelle_fs', 'libelle_type_structure', 'libelle_type_prestation', 'verifierIncoherenceFormationSanitaire']
    ].drop_duplicates().head(10)
    print(anomaly_examples.to_string())
else:
    print(f"\n  ✅ Aucune incohérence détectée!")

print(f"\n✅ Colonne créée: verifierIncoherenceFormationSanitaire (0=normal, 1=incohérence)")

VÉRIFICATION CSPS BÉNÉFICIANT D'ACTES SPÉCIALISÉS


KeyError: 'id_type_structure'

In [117]:
# ==============================================================
# Qualité des données transmises (irrégularités, incohérences) (verifierIncoherenceSexe): Algo 2 et 3
 #- Sexe masculin bénéficiant de : accouchements, interventions obstétricales,
#soins pendant la grossesse, PF (hormis vasectomie et condom masculin), dépistage + traitement des lésions précancéreuses
# - Sexe féminin bénéficiant de : vasectomie
# ==============================================================

print("="*70)
print(" VEERIFICATION INCOHERENCE SEXE")
print("="*70)

# Condition 1: (sexe = 'male') ET consultation_type IN (1, 2, 3, 5) ET type_prestation IN (23, 31)
condition_male = (
    (df['sex'].isin(['male'])) &
    (df['consultation_type'].isin([1, 2, 3, 5])) &
    (df['type_prestation'].isin([23, 31]))
)

# Condition 2: (sexe = 'female') ET consultation_type = 23
condition_female = (
    (df['sex'].isin(['female'])) &
    (df['consultation_type'] == 23)
)

# Combiner les deux conditions avec OR
df['verifierIncoherenceSexe'] = (condition_male | condition_female).astype(int)

# Statistiques
criteria_met = df['verifierIncoherenceSexe'].sum()
criteria_not_met = len(df) - criteria_met

print(f"\n RÉSULTAT:")
print(f"  ✓ Critères respectés: {criteria_met}")
print(f"  ✗ Critères non respectés: {criteria_not_met}")

# Afficher les exemples
if criteria_met > 0:
    print(f"\n  Exemples (critères respectés):")
    examples = df[df['verifierIncoherenceSexe'] == 1][
        ['sex', 'consultation_type', 'type_prestation', 'verifierIncoherenceSexe']
    ].drop_duplicates().head(10)
    print(examples.to_string())

print(f"\n✅ Colonne créée: verifierIncoherenceSexe (0=non, 1=oui)")

 VEERIFICATION INCOHERENCE SEXE

 RÉSULTAT:
  ✓ Critères respectés: 0
  ✗ Critères non respectés: 1433

✅ Colonne créée: verifierIncoherenceSexe (0=non, 1=oui)


In [118]:
# ==============================================================
# DÉTECTION DES FACTURES AVEC COÛTS D'ÉVACUATION ÉLEVÉS (>= 120000 F) (verifierMontantEvacuation)
# ==============================================================

print("="*70)
print("DÉTECTION DES COÛTS D'ÉVACUATION ÉLEVÉS (>= 120000 F)")
print("="*70)

# Créer le flag: 1 si cout_evacuation >= 120000, 0 sinon
df['verifierMontantEvacuation'] = (df['cout_evacuation'] >= 120000).astype(int)

# Statistiques
high_cost_count = df['verifierMontantEvacuation'].sum()
normal_cost_count = len(df) - high_cost_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Coûts normaux (< 120000 F): {normal_cost_count}")
print(f"  ⚠️  Coûts élevés (>= 120000 F): {high_cost_count}")

# Afficher les exemples de coûts élevés
if high_cost_count > 0:
    print(f"\n  Exemples de factures avec coûts d'évacuation élevés:")
    high_cost_examples = df[df['verifierMontantEvacuation'] == 1][
        ['nom_patient', 'cout_evacuation', 'verifierMontantEvacuation']
    ].drop_duplicates().head(10)
    print(high_cost_examples.to_string())

print(f"\n✅ Colonne créée: verifierMontantEvacuation (0=normal, 1=coût >= 120000 F)")

DÉTECTION DES COÛTS D'ÉVACUATION ÉLEVÉS (>= 120000 F)

 RÉSULTAT:
  ✓ Coûts normaux (< 120000 F): 1429
  ⚠️  Coûts élevés (>= 120000 F): 4

  Exemples de factures avec coûts d'évacuation élevés:
         nom_patient  cout_evacuation  verifierMontantEvacuation
228  BIRBA ELISABETH         120000.0                          1
229       HIEN TODAO         160000.0                          1
230   SANKARA  NATOU         160000.0                          1
231     SOMDA BALTAI         160000.0                          1

✅ Colonne créée: verifierMontantEvacuation (0=normal, 1=coût >= 120000 F)


In [119]:
# ============================================================== 
# VÉRIFICATION (verifierHospitalisationEtEvacuation): bénéficiaires, Dépistage+traitement des lésions
# précancéreuses bénéficiant des biens/services : hospitalisation et évacuation
# ==============================================================

print("="*70)
print("VÉRIFICATION DÉPISTAGE+TRAITEMENT/HOSPITALISATION ET ÉVACUATION")
print("="*70)

# Créer le flag basé sur les conditions:
# Si consultation_type = 3 & cout_evacuation > 0 & type_hospitalisation = true
# → 1 (critères respectés), 0 sinon

# Note: "type_hospitalisation = true" correspond à type_observation != 'ambulatoire'
df['verifierHospitalisationEtEvacuation'] = (
    (df['consultation_type'] == 3) &  # consultation_type = 3
    (df['cout_evacuation'] > 0) &      # cout_evacuation > 0
    (df['type_observation'] != 'ambulatoire')  # type_hospitalisation = true
).astype(int)

# Statistiques
criteria_met = df['verifierHospitalisationEtEvacuation'].sum()
criteria_not_met = len(df) - criteria_met

print(f"\n RÉSULTAT:")
print(f"  ✓ Critères respectés: {criteria_met}")
print(f"  ✗ Critères non respectés: {criteria_not_met}")

# Afficher les exemples qui respectent tous les critères
if criteria_met > 0:
    print(f"\n  Exemples (critères respectés):")
    examples = df[df['verifierHospitalisationEtEvacuation'] == 1][
        ['nom_patient', 'consultation_type', 'cout_evacuation', 'type_observation', 'verifierHospitalisationEtEvacuation']
    ].drop_duplicates().head(10)
    print(examples.to_string())
print(f"\n✅ Colonne créée: verifierHospitalisationEtEvacuation (0=non, 1=oui)")

VÉRIFICATION DÉPISTAGE+TRAITEMENT/HOSPITALISATION ET ÉVACUATION

 RÉSULTAT:
  ✓ Critères respectés: 0
  ✗ Critères non respectés: 1433

✅ Colonne créée: verifierHospitalisationEtEvacuation (0=non, 1=oui)


In [120]:
# ==============================================================
# VÉRIFICATION (verifierEvacuationIncoherence): 
# Factures avec mode_sortie différent de "Evacué" (58) et cout_evacuation > 0
# ==============================================================

print("="*70)
print("VÉRIFICATION INCOHÉRENCE ÉVACUATION")
print("="*70)

# Créer le flag basé sur les conditions:
# Si mode_sortie != 58 ET cout_evacuation > 0 → 1 (incohérence détectée), 0 sinon

df['verifierEvacuationIncoherence'] = (
    (df['mode_sortie'] != 58) & 
    (df['cout_evacuation'] > 0)
).astype(int)

# Statistiques
anomaly_count = df['verifierEvacuationIncoherence'].sum()
normal_count = len(df) - anomaly_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Cas cohérents (mode_sortie=58 ou cout_evacuation=0): {normal_count}")
print(f"  ⚠️  Incohérences détectées (mode_sortie!=58 et cout_evacuation>0): {anomaly_count}")

# Afficher les exemples d'incohérences
if anomaly_count > 0:
    print(f"\n  Exemples d'incohérences:")
    anomaly_examples = df[df['verifierEvacuationIncoherence'] == 1][
        ['nom_patient', 'mode_sortie', 'cout_evacuation', 'verifierEvacuationIncoherence']
    ].drop_duplicates().head(10)
    print(anomaly_examples.to_string())

print(f"\n✅ Colonne créée: verifierEvacuationIncoherence (0=cohérent, 1=incohérent)")

VÉRIFICATION INCOHÉRENCE ÉVACUATION

 RÉSULTAT:
  ✓ Cas cohérents (mode_sortie=58 ou cout_evacuation=0): 1433
  ⚠️  Incohérences détectées (mode_sortie!=58 et cout_evacuation>0): 0

✅ Colonne créée: verifierEvacuationIncoherence (0=cohérent, 1=incohérent)


In [121]:
# ==============================================================
# VÉRIFICATION (verifierHospitalisation): Bénéficiaires PF + Patients ambulatoires
# Condition: (consultation_type = planification familiale OR type_prestation 
# IN (Soins curatifs en ambulatoire enfant=20, Soins curatifs en ambulatoire grossesse=35))
# AND type_observation IS NOT NULL
# ==============================================================

print("="*70)
print("VÉRIFICATION BÉNÉFICIAIRES PF ou PATIENTS AMBULATOIRES BENEFICIANTS DE L'HOSPITALISATION ")
print("="*70)

# Créer le flag basé sur les conditions:
# Si (consultation_type = 5 OU type_prestation IN (20, 35)) ET type_observation != null
# → 1 (critères respectés), 0 sinon

df['verifierHopitalisation_PF_Ambulatoires'] = (
    ((df['consultation_type'] == 5) | (df['type_prestation'].isin([20, 35]))) &
    (df['type_observation'].notna() & df['type_observation']!= 'ambulatoire')
).astype(int)

# Statistiques
criteria_met = df['verifierHopitalisation_PF_Ambulatoires'].sum()
criteria_not_met = len(df) - criteria_met

print(f"\n RÉSULTAT:")
print(f"  ✓ Critères respectés: {criteria_met}")
print(f"  ✗ Critères non respectés: {criteria_not_met}")

# Afficher les exemples qui respectent tous les critères
if criteria_met > 0:
    print(f"\n  Exemples (critères respectés):")
    examples = df[df['verifierHopitalisation_PF_Ambulatoires'] == 1][
        ['nom_patient', 'consultation_type', 'type_prestation', 'type_observation', 'verifierHopitalisation_PF_Ambulatoires']
    ].drop_duplicates().head(10)
    print(examples.to_string())

print(f"\n✅ Colonne créée: verifierHopitalisation_PF_Ambulatoires (0=non, 1=oui)")

VÉRIFICATION BÉNÉFICIAIRES PF ou PATIENTS AMBULATOIRES BENEFICIANTS DE L'HOSPITALISATION 

 RÉSULTAT:
  ✓ Critères respectés: 1010
  ✗ Critères non respectés: 423

  Exemples (critères respectés):
              nom_patient  consultation_type  type_prestation  type_observation  verifierHopitalisation_PF_Ambulatoires
1        KISWEFO RAFIATOU                4.0             20.0               NaN                                       1
2             OUALI TALAD                4.0             20.0               NaN                                       1
3        BAGAGNAN YASMINA                4.0             20.0               NaN                                       1
5         FANTAKOY AROUNA                4.0             20.0               NaN                                       1
8   BONKOUNGOU MARGUERITE                5.0             29.0               NaN                                       1
10      SAWADOGO SALIMATA                5.0             30.0               NaN    

In [122]:
# ==============================================================
# VÉRIFICATION (verifierPrestationEnfant): Enfants < 9 ans bénéficiant de prestations spécifiques
# Consultation types: 1=Accouchements+interventions obstétricales, 
#                    2=Soins pendant la grossesse, 
#                    3=PF (Planification familiale),
#                    5=Dépistage+traitement lésions précancéreuses
# ==============================================================

print("="*70)
print("VÉRIFICATION ENFANTS < 9 ANS BÉNÉFICIANT DE PRESTATIONS SPÉCIFIQUES")
print("="*70)

# Créer le flag basé sur les conditions:
# Si age_patient_calculated < 9 ET consultation_type IN (1, 2, 3, 5)
# → 1 (anomalie détectée), 0 sinon

df['verifierPrestationEnfant'] = (
    (df['age_patient_calculated'] < 9) &
    (df['consultation_type'].isin([1, 2, 3, 5]))
).astype(int)

# Statistiques
anomaly_count = df['verifierPrestationEnfant'].sum()
normal_count = len(df) - anomaly_count

print(f"\n RÉSULTAT:")
print(f"  ✓ Cas normaux (age >= 9 ou consultation_type non spécifique): {normal_count}")
print(f"  ⚠️  Anomalies (enfant < 9 ans avec prestation spécifique): {anomaly_count}")

# Afficher les exemples d'anomalies
if anomaly_count > 0:
    print(f"\n  Exemples d'anomalies (enfants < 9 ans):")
    anomaly_examples = df[df['verifierPrestationEnfant'] == 1][
        ['nom_patient', 'age_patient_calculated', 'consultation_type', 'verifierPrestationEnfant']
    ].drop_duplicates().head(10)
    print(anomaly_examples.to_string())

print(f"\n✅ Colonne créée: verifierPrestationEnfant (0=normal, 1=anomalie détectée)")

VÉRIFICATION ENFANTS < 9 ANS BÉNÉFICIANT DE PRESTATIONS SPÉCIFIQUES

 RÉSULTAT:
  ✓ Cas normaux (age >= 9 ou consultation_type non spécifique): 1433
  ⚠️  Anomalies (enfant < 9 ans avec prestation spécifique): 0

✅ Colonne créée: verifierPrestationEnfant (0=normal, 1=anomalie détectée)


In [123]:
# ==============================================================
# Création des flags pour le colonnes quantité et couts et nbre_jours (existe ou non)
# ==============================================================
num_colonne = [
    
    "quantite_total_prod", "quantite_total_act", "quantite_total_ex",
    "cout_total_prod", "cout_total_act", "cout_total_ex",
    "cout_mise_en_observation", "cout_evacuation","nbre_jours"
]

# Create existence flags for numeric columns
for col in num_colonne:
    if col in df.columns:
        flag_col = f"{col}_exists"
        df[flag_col] = df[col].notna().astype(int)
        exists_count = df[flag_col].sum()
        print(f"✓ {flag_col}: {exists_count} non-null, {len(df) - exists_count} null")

print("\n✅ Existence flags created for all numeric columns")
print([c for c in df.columns if "_exists" in c])

✓ quantite_total_prod_exists: 1433 non-null, 0 null
✓ quantite_total_act_exists: 1433 non-null, 0 null
✓ quantite_total_ex_exists: 231 non-null, 1202 null
✓ cout_total_prod_exists: 1433 non-null, 0 null
✓ cout_total_act_exists: 1433 non-null, 0 null
✓ cout_total_ex_exists: 1433 non-null, 0 null
✓ cout_mise_en_observation_exists: 1433 non-null, 0 null
✓ cout_evacuation_exists: 1433 non-null, 0 null
✓ nbre_jours_exists: 1433 non-null, 0 null

✅ Existence flags created for all numeric columns
['quantite_total_prod_exists', 'quantite_total_act_exists', 'quantite_total_ex_exists', 'cout_total_prod_exists', 'cout_total_act_exists', 'cout_total_ex_exists', 'cout_mise_en_observation_exists', 'cout_evacuation_exists', 'nbre_jours_exists']


In [124]:
print(" liste des colonnes \n",df.columns)

 liste des colonnes 
 Index(['nom_patient', 'village', 'distance_village', 'age_patient', 'sex',
       'visit_date', 'registre_number', 'consultation_type', 'type_prestation',
       'quantite_total_prod', 'quantite_total_act', 'quantite_total_ex',
       'cout_total_prod', 'cout_total_act', 'cout_total_ex',
       'type_observation', 'nbre_jours', 'cout_mise_en_observation',
       'cout_evacuation', 'created_at', 'id_prescripteur', 'id_gerant',
       'date_entree', 'date_sortie', 'mode_sortie', 'status_verification',
       'observations_verification', 'date_verification',
       'nom_patient_is_filled', 'distance_village_is_filled',
       'age_patient_is_filled', 'registre_number_is_filled',
       'id_prescripteur_is_filled', 'id_gerant_is_filled', 'sex_is_filled',
       'consultation_type_is_filled', 'type_prestation_is_filled',
       'visit_date_is_filled', 'age_patient_date_naissance',
       'age_patient_calculated', 'age_patient_is_valid', 'sex_is_valid',
       'date_ent

In [125]:
df.head()

,nom_patient,village,distance_village,age_patient,sex,visit_date,registre_number,consultation_type,type_prestation,quantite_total_prod,...,verifierPrestationEnfant,quantite_total_prod_exists,quantite_total_act_exists,quantite_total_ex_exists,cout_total_prod_exists,cout_total_act_exists,cout_total_ex_exists,cout_mise_en_observation_exists,cout_evacuation_exists,nbre_jours_exists
0,NaN,NaN,1.0,NaN,female,2025-09-22,359,1.0,34.0,32.0,...,0,1,1,0,1,1,1,1,1,1
1,KISWEFO RAFIATOU,NaN,1.0,NaN,female,2025-09-15,NaN,4.0,20.0,25.0,...,0,1,1,0,1,1,1,1,1,1
2,OUALI TALAD,NaN,1.0,NaN,female,2025-08-29,NaN,4.0,20.0,1.0,...,0,1,1,0,1,1,1,1,1,1
3,BAGAGNAN YASMINA,NaN,1.0,NaN,female,2025-08-29,NaN,4.0,20.0,1.0,...,0,1,1,1,1,1,1,1,1,1
4,KISWEFO RAFIATOU,NaN,1.0,NaN,female,2025-08-26,NaN,4.0,20.0,25.0,...,0,1,1,0,1,1,1,1,1,1


In [126]:
df.columns

Index(['nom_patient', 'village', 'distance_village', 'age_patient', 'sex',
       'visit_date', 'registre_number', 'consultation_type', 'type_prestation',
       'quantite_total_prod', 'quantite_total_act', 'quantite_total_ex',
       'cout_total_prod', 'cout_total_act', 'cout_total_ex',
       'type_observation', 'nbre_jours', 'cout_mise_en_observation',
       'cout_evacuation', 'created_at', 'id_prescripteur', 'id_gerant',
       'date_entree', 'date_sortie', 'mode_sortie', 'status_verification',
       'observations_verification', 'date_verification',
       'nom_patient_is_filled', 'distance_village_is_filled',
       'age_patient_is_filled', 'registre_number_is_filled',
       'id_prescripteur_is_filled', 'id_gerant_is_filled', 'sex_is_filled',
       'consultation_type_is_filled', 'type_prestation_is_filled',
       'visit_date_is_filled', 'age_patient_date_naissance',
       'age_patient_calculated', 'age_patient_is_valid', 'sex_is_valid',
       'date_entree_is_valid', 'date_s

In [127]:
# Supprimer les colonnes demandées si elles existent
cols_to_remove = [
    'nom_patient', 'village', 'distance_village', 'age_patient', 'sex', 'visit_date',
    'registre_number', 'quantite_total_prod', 'quantite_total_act', 'quantite_total_ex',
    'cout_total_prod', 'cout_total_act', 'cout_total_ex', 'nbre_jours', 'created_at',
    'id_prescripteur', 'id_gerant', 'date_entree', 'date_sortie', 'date_verification',
    'cout_evacuation', 'cout_mise_en_observation','type_observation_filled',
    'age_patient_date_naissance','libelle_type_prestation','libelle_type_prestation', 
    'libelle_fs', 'libelle_type_structure','age_patient_date_naissance',
       'age_patient_calculated','libelle_type_prestation', 'libelle_fs'
]

dfforml=df.copy()
cols_present = [c for c in cols_to_remove if c in dfforml.columns]
dfforml.drop(columns=cols_present, inplace=True)
print(f"Colonnes supprimées: {cols_present}")
print(f"Nouveau shape: {dfforml.shape}")
print(dfforml.columns)

Colonnes supprimées: ['nom_patient', 'village', 'distance_village', 'age_patient', 'sex', 'visit_date', 'registre_number', 'quantite_total_prod', 'quantite_total_act', 'quantite_total_ex', 'cout_total_prod', 'cout_total_act', 'cout_total_ex', 'nbre_jours', 'created_at', 'id_prescripteur', 'id_gerant', 'date_entree', 'date_sortie', 'date_verification', 'cout_evacuation', 'cout_mise_en_observation', 'age_patient_date_naissance', 'age_patient_date_naissance', 'age_patient_calculated']
Nouveau shape: (1433, 40)
Index(['consultation_type', 'type_prestation', 'type_observation',
       'mode_sortie', 'status_verification', 'observations_verification',
       'nom_patient_is_filled', 'distance_village_is_filled',
       'age_patient_is_filled', 'registre_number_is_filled',
       'id_prescripteur_is_filled', 'id_gerant_is_filled', 'sex_is_filled',
       'consultation_type_is_filled', 'type_prestation_is_filled',
       'visit_date_is_filled', 'age_patient_is_valid', 'sex_is_valid',
       'd

In [128]:
dfforml.head()

,consultation_type,type_prestation,type_observation,mode_sortie,status_verification,observations_verification,nom_patient_is_filled,distance_village_is_filled,age_patient_is_filled,registre_number_is_filled,...,verifierPrestationEnfant,quantite_total_prod_exists,quantite_total_act_exists,quantite_total_ex_exists,cout_total_prod_exists,cout_total_act_exists,cout_total_ex_exists,cout_mise_en_observation_exists,cout_evacuation_exists,nbre_jours_exists
0,1.0,34.0,NaN,59.0,a_corriger,Absence de: nom_patient,0,1,0,1,...,0,1,1,0,1,1,1,1,1,1
1,4.0,20.0,NaN,59.0,a_corriger,Absence de: registre_number,1,1,0,0,...,0,1,1,0,1,1,1,1,1,1
2,4.0,20.0,NaN,59.0,a_corriger,Absence de: registre_number,1,1,0,0,...,0,1,1,0,1,1,1,1,1,1
3,4.0,20.0,NaN,59.0,a_corriger,Absence de: registre_number,1,1,0,0,...,0,1,1,1,1,1,1,1,1,1
4,4.0,20.0,NaN,59.0,a_corriger,Absence de: registre_number,1,1,0,0,...,0,1,1,0,1,1,1,1,1,1


In [129]:
dfforml.to_csv("data/dfforml2.csv", index=False, encoding='utf-8')
print("✅ DataFrame exported to 'dfforml.csv'")
print(f"Shape: {dfforml.shape}")

✅ DataFrame exported to 'dfforml.csv'
Shape: (1433, 40)
